In [1]:
!pip install rapidfuzz requests beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 17.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import requests
import re
import json
import pandas as pd
from rapidfuzz import process, fuzz
from collections import defaultdict, Counter

Mounted at /content/drive


**Scrapping de Wikipedia MG**

In [8]:
def fetch_wikipedia_mg(nb_articles=200):
    mots    = set()
    phrases = []
    url     = "https://mg.wikipedia.org/w/api.php"

    # ── Header obligatoire pour Wikipedia ─────────────────────────────
    headers = {
        "User-Agent": "NLPMalagasyBot/1.0 (projet etudiant NLP; contact: email@gmail.com)"
    }

    # ── Étape 1 : récupère les titres de pages ─────────────────────────
    try:
        r = requests.get(url, headers=headers, params={
            "action": "query", "list": "random",
            "rnnamespace": 0, "rnlimit": nb_articles,
            "format": "json"
        }, timeout=15)

        if r.status_code != 200:
            print(f"[Wikipedia] Erreur HTTP {r.status_code} : {r.text[:100]}")
            return mots, phrases

        pages = r.json()["query"]["random"]

    except Exception as e:
        print(f"[Wikipedia] Erreur récupération titres : {e}")
        return mots, phrases

    # ── Étape 2 : récupère le contenu de chaque page ───────────────────
    ok, echec = 0, 0

    for page in pages:
        try:
            rd = requests.get(url, headers=headers, params={
                "action": "query", "pageids": page["id"],
                "prop": "extracts", "explaintext": True,
                "format": "json"
            }, timeout=10)

            if rd.status_code != 200 or not rd.text.strip():
                echec += 1
                continue

            texte = rd.json()["query"]["pages"][str(page["id"])].get("extract", "")

            if not texte.strip():
                echec += 1
                continue

            for phrase in re.split(r'[.!?\n]+', texte):
                phrase = phrase.strip()
                if 10 < len(phrase) < 300:
                    phrases.append(phrase)

            for mot in re.findall(r"[a-zA-Zñ']{2,}", texte.lower()):
                mots.add(mot)

            ok += 1

        except Exception:
            echec += 1
            continue

    print(f"[Wikipedia MG] {ok} pages OK | {echec} échecs")
    print(f"               {len(mots)} mots | {len(phrases)} phrases")
    return mots, phrases


mots_wiki, phrases_wiki = fetch_wikipedia_mg(nb_articles=200)

[Wikipedia MG] 200 pages OK | 0 échecs
               2589 mots | 2091 phrases


**Scrapping de Bible Malagasy**

In [9]:
def fetch_bible_malagasy():
    """
    Télécharge et parse la Bible en malagasy depuis une source libre.
    Retourne : (set de mots, liste de phrases)
    """
    mots    = set()
    phrases = []

    # Source : Projet Gutenberg / NLTK corpus (Bible MG en texte brut)
    url = "https://raw.githubusercontent.com/christos-c/bible-corpus/master/bibles/Malagasy.xml"

    try:
        r = requests.get(url, timeout=30)
        # Les versets sont dans des balises <seg> ou le texte brut entre balises
        # On extrait tout le texte entre balises avec regex simple
        texte_brut = re.sub(r'<[^>]+>', ' ', r.text)

        for ligne in texte_brut.split('\n'):
            ligne = ligne.strip()
            if len(ligne) > 10:
                phrases.append(ligne)
                for mot in re.findall(r"[a-zA-Zñ']{2,}", ligne.lower()):
                    mots.add(mot)

        print(f"[Bible MG] {len(mots)} mots | {len(phrases)} phrases")

    except Exception as e:
        print(f"[Bible MG] Erreur : {e}")
        # Fallback : fichier local depuis Drive si tu l'as
        # texte_brut = open("/content/drive/MyDrive/bible_mg.txt").read()

    return mots, phrases


mots_bible, phrases_bible = fetch_bible_malagasy()

[Bible MG] 26616 mots | 31105 phrases


**Fusion et nettoyage**

In [10]:
def nettoyer_mot(mot: str) -> str:
    """Normalise un mot : minuscules, strip, supprime apostrophes isolées."""
    mot = mot.lower().strip().strip("'")
    return mot

def est_mot_valide(mot: str) -> bool:
    """
    Filtre les mots bruit :
    - longueur entre 2 et 25 caractères
    - contient au moins une voyelle malagasy
    - pas que des chiffres ou caractères spéciaux
    """
    VOYELLES_MG = set("aeiou")
    if not (2 <= len(mot) <= 25):
        return False
    if not any(c in VOYELLES_MG for c in mot):
        return False
    if not re.match(r"^[a-zA-Zñ']+$", mot):
        return False
    return True


# ── Fusion des deux sources ────────────────────────────────────────────
tous_les_mots = mots_wiki | mots_bible
toutes_phrases = phrases_wiki + phrases_bible

# ── Nettoyage du dictionnaire ──────────────────────────────────────────
DICTIONNAIRE = set()
for mot in tous_les_mots:
    mot_propre = nettoyer_mot(mot)
    if est_mot_valide(mot_propre):
        DICTIONNAIRE.add(mot_propre)

# ── Nettoyage des phrases ──────────────────────────────────────────────
CORPUS_PHRASES = []
for phrase in toutes_phrases:
    phrase = phrase.strip()
    # Garde les phrases avec au moins 3 mots
    if len(phrase.split()) >= 3:
        CORPUS_PHRASES.append(phrase)

# Supprime les doublons de phrases
CORPUS_PHRASES = list(dict.fromkeys(CORPUS_PHRASES))

print(f"\n✅ DICTIONNAIRE final : {len(DICTIONNAIRE)} mots")
print(f"✅ CORPUS PHRASES     : {len(CORPUS_PHRASES)} phrases")
print(f"\nExemples de mots : {list(DICTIONNAIRE)[:15]}")
print(f"\nExemples de phrases :")
for p in CORPUS_PHRASES[:5]:
    print(f"  → {p}")


✅ DICTIONNAIRE final : 28286 mots
✅ CORPUS PHRASES     : 32083 phrases

Exemples de mots : ['nijangajanga', 'fisoronana', 'zanakai', 'fitahirizana', 'azynanao', 'fampahatahorako', 'lambanay', 'hiatrika', 'hampanaoviny', 'novoriko', 'afatory', "tokan'andriamanitra", 'habakoka', 'voarava', 'famatorany']

Exemples de phrases :
  → 3222°W﻿ / 44
  → I Béguey dia kaominina ao amin'ny fivondronan'i Bordeaux, ao amin'ny departemantan'i Gironde, ao amin'ny faritr'i Nouvelle-Aquitaine, ao Frantsa
  → Ny INSEE dia mampiasa ny kaodim-paositra 33040
  → I Jean Rupert no ben'ny tanàna mandritry ny taona 2008–2014
  → Ilay kaominina dia kaominina mpikambana amin'ny fivondronan-kaominin'i Coteaux de Garonne


**Sauvegarde dans Drive**

In [11]:
DOSSIER = "/content/drive/MyDrive/"  # ← adapte si sous-dossier

with open(DOSSIER + "dictionnaire_mg.json", "w", encoding="utf-8") as f:
    json.dump(list(DICTIONNAIRE), f, ensure_ascii=False)

with open(DOSSIER + "corpus_phrases_mg.json", "w", encoding="utf-8") as f:
    json.dump(CORPUS_PHRASES, f, ensure_ascii=False)

print("✅ Fichiers sauvegardés dans Drive")

✅ Fichiers sauvegardés dans Drive


**Tokenisation**

In [12]:
STOPWORDS_MG = {
    "ny", "sy", "no", "na", "fa", "ka", "dia", "ary",
    "ao", "an", "am", "in", "ho", "de", "aza", "ve",
    "re", "tsy", "amin", "avy", "any", "eny", "izay",
    "izy", "aho", "ianao", "isika", "ilay", "ireo",
    "ity", "io", "iny", "koa", "ihany", "foana", "nefa",
    "saingy", "kanefa", "raha", "satria", "mba", "aza",
}

def tokeniser(texte: str) -> list:
    """
    Tokeniseur malagasy qui gère :
    - Le trait d'union '-' (fampianaran-teny → 2 tokens)
    - L'apostrophe "'" (amin'ny → 2 tokens)
    - La ponctuation en fin/début de token
    """
    # Normalise les apostrophes typographiques
    texte = texte.replace("\u2019", "'").replace("\u2018", "'")

    tokens = []
    # Split initial sur espaces et ponctuation forte
    for bloc in re.split(r'[\s,;:!?."()\[\]]+', texte):
        bloc = bloc.strip()
        if not bloc:
            continue
        # Split sur trait d'union
        for partie in bloc.split("-"):
            # Split sur apostrophe
            for sp in partie.split("'"):
                sp = sp.lower().strip().strip("'")
                if len(sp) >= 2:
                    tokens.append(sp)

    return tokens

**Module 1 : Correcteur orthographique**

In [13]:
DICO_LIST = list(DICTIONNAIRE)  # pour rapidfuzz

def est_correct(mot: str) -> bool:
    return mot.lower().strip() in DICTIONNAIRE

def suggerer(mot: str, nb=3, seuil=70) -> list:
    resultats = process.extract(
        mot.lower().strip(),
        DICO_LIST,
        scorer=fuzz.ratio,
        limit=nb
    )
    return [(m, s) for m, s, _ in resultats if s >= seuil]

def corriger_texte(texte: str) -> dict:
    """Retourne {mot_incorrect: [(suggestion, score), ...]}"""
    erreurs = {}
    for mot in set(tokeniser(texte)):
        if mot in STOPWORDS_MG:
            continue
        if not est_correct(mot):
            erreurs[mot] = suggerer(mot)
    return erreurs


# ── Test ───────────────────────────────────────────────────────────────
print("=== Module 1 : Correcteur ===\n")
tests = [
    "Ny fitiavana sy ny fampianaran-teny no zava-dehibe",
    "Amin'ny alina dia matory ny ankizy malagasi",
]
for t in tests:
    print(f"📝 {t}")
    erreurs = corriger_texte(t)
    if not erreurs:
        print("   ✅ Aucune erreur")
    for mot, sugg in erreurs.items():
        print(f"   ❌ '{mot}' → {[s for s,_ in sugg] or ['—']}")
    print()

=== Module 1 : Correcteur ===

📝 Ny fitiavana sy ny fampianaran-teny no zava-dehibe
   ✅ Aucune erreur

📝 Amin'ny alina dia matory ny ankizy malagasi
   ❌ 'malagasi' → ['malagasy', 'malanga', 'alagoas']



**Module 2: Verification par REGEX/Phonotactique**

In [15]:
from dataclasses import dataclass
from typing import List

@dataclass
class ErreurPhono:
    mot: str
    regle: str
    description: str

REGLES_PHONOTACTIQUES = [
    {
        "id": "COMB_INTERDIT",
        "pattern": re.compile(r'nb|mk|dt|bp|sz', re.I),
        "description": "Combinaison de consonnes inexistante en malagasy"
    },
    {
        "id": "NK_DEBUT",
        "pattern": re.compile(r'^nk|nb|y|ml|nt|np', re.I),
        "description": "Début de mot  interdit"
    },
    {
        "id": "LETTRE_ETRANGERE",
        "pattern": re.compile(r'[cquwx]', re.I),
        "description": "Lettre absente de l'alphabet malagasy (c, q, u, w, x)"
    },
    {
        "id": "DOUBLE_CONSONNE",
        "pattern": re.compile(r'([bcdfgjklmnprstvz])\1', re.I),
        "description": "Double consonne inhabituelle"
    },
    {
        "id": "FIN_CONSONNE",
        "pattern": re.compile(r'[bcdfgjklmprstv]$', re.I),
        "description": "Fin de mot atypique (malagasy finit par voyelle ou -ny)"
    },
]

def verifier_phonotactique(mot: str) -> List[ErreurPhono]:
    # Si le mot est dans le dictionnaire, il est correct par définition
    if mot.lower() in DICTIONNAIRE:
        return []
    erreurs = []
    for regle in REGLES_PHONOTACTIQUES:
        if regle["pattern"].search(mot.strip()):
            erreurs.append(ErreurPhono(mot, regle["id"], regle["description"]))
    return erreurs


# ── Test ───────────────────────────────────────────────────────────────
print("=== Module 2 : Phonotactique ===\n")
tests_phono = ["manosika", "ankizy", "nbato", "foxer", "mkcola", "trano"]
for mot in tests_phono:
    e = verifier_phonotactique(mot)
    if e:
        print(f"  ❌ '{mot}' → [{e[0].regle}] {e[0].description}")
    else:
        print(f"  ✅ '{mot}'")

=== Module 2 : Phonotactique ===

  ✅ 'manosika'
  ✅ 'ankizy'
  ❌ 'nbato' → [COMB_INTERDIT] Combinaison de consonnes inexistante en malagasy
  ❌ 'foxer' → [LETTRE_ETRANGERE] Lettre absente de l'alphabet malagasy (c, q, u, w, x)
  ❌ 'mkcola' → [COMB_INTERDIT] Combinaison de consonnes inexistante en malagasy
  ✅ 'trano'


**Module 3: Lemmatisation**

In [37]:
# Tables morphologiques du malagasy
PREFIXES = [
    "mpampi", "mpanao", "mpan", "mampian", "mampi", "mamp",
    "mifan", "maha", "man", "mam", "mi", "ma",
    "fampi", "famp", "fan", "fam", "fi", "fa",
    "tamin", "tan", "am", "an", "ifan", "i","m"
]
SUFFIXES = ["andriny", "indriny", "iana", "iny", "ana", "ina", "na", "ny"]

MUTATIONS = {
    "d":  ["r", "l", "dr"],   # drazana → razana, danja → lanja
    "j":  ["z", "j"],         # jaza → zaza
    "nd": ["l", "r", "nd"],   # ndotra → lotra
    "dr": ["r"],               # dresy → resy
    "b":  ["v", "b"],          # bory → vory
    "mp": ["p", "mp"],         # mpanao → garder mp (préfixe)
}

def appliquer_mutations(mot: str) -> list:
    """
    Génère toutes les racines candidates en appliquant
    les mutations consonantiques inverses.

    Ex: "drazana" → ["rrazana"(invalide), "lrazana"(invalide), "razana" ✅]
    """
    candidats = []
    mot_lower = mot.lower().strip()

    for mutation, origines in MUTATIONS.items():
        if mot_lower.startswith(mutation):
            reste = mot_lower[len(mutation):]  # ce qui suit la consonne mutée
            for origine in origines:
                candidat = origine + reste
                if candidat in DICTIONNAIRE:
                    candidats.append(candidat)

    return candidats


# ── Racines qui se terminent naturellement par une voyelle ────────────
# Le suffixe -na est valide SEULEMENT si la racine sans -na
# n'existe pas mais la racine avec -na existe dans le dico

def suffixe_na_est_valide(mot: str, racine_candidate: str) -> bool:
    """
    Vérifie si retirer '-na' est justifié.
    Règle : on retire -na seulement si la racine candidate
    n'existe PAS dans le dico telle quelle.
    Ex : "alina" → racine candidate "ali" → "ali" absent du dico → on garde "alina"
    Ex : "fomba" → "fomba" présent → on peut retirer -na de "fomban" → "fomba" ✅
    """
    return racine_candidate in DICTIONNAIRE


def restaurer_voyelle_finale(mot: str) -> str:
    """
    Restaure la voyelle finale coupée par une apostrophe ou liaison.
    Ex : "toetr" → essaie "toetra", "toetre", "toetri", "toetro"
    Ex : "fomban" → essaie "fomba" (retire -n de liaison)
    """
    voyelles = ["a", "e", "i", "o"]

    # Cas 1 : mot finit par consonne → essaie d'ajouter une voyelle
    if mot and mot[-1] not in "aeiou":
        # Retire d'abord le -n de liaison si présent
        base = mot[:-1] if mot.endswith("n") else mot
        for v in voyelles:
            candidat = base + v
            if candidat in DICTIONNAIRE:
                return candidat
        # Essaie aussi avec juste une voyelle ajoutée (sans retirer -n)
        for v in voyelles:
            candidat = mot + v
            if candidat in DICTIONNAIRE:
                return candidat

    return mot  # pas de restauration possible

def est_correct(mot: str) -> bool:
    """
    Vérifie si un mot est correct.
    Tente aussi la restauration de voyelle finale avant de déclarer inconnu.

    """
    mot_c = mot.lower().strip()

    # Vérification directe
    if mot_c in DICTIONNAIRE:
        return True

    # Tentative de restauration de voyelle finale
    restaure = restaurer_voyelle_finale(mot_c)
    if restaure != mot_c and restaure in DICTIONNAIRE:
        return True

    return False

def lemmatiser(mot: str) -> dict:
    mot_c = mot.lower().strip()

    candidats = []  # (racine, score_strip, methode)

    # ── Étape 0 : restauration voyelle finale (toetr→toetra, fomban→fomba)
    restaure = restaurer_voyelle_finale(mot_c)
    if restaure != mot_c:
        # On travaille aussi sur la forme restaurée
        formes_a_analyser = [mot_c, restaure]
    else:
        formes_a_analyser = [mot_c]

    for forme in formes_a_analyser:

        # ── N1 : règles morphologiques (préfixes + suffixes) ──────────
        for pfx in sorted(PREFIXES, key=len, reverse=True):
            if forme.startswith(pfx) and len(forme) > len(pfx) + 2:
                reste = forme[len(pfx):]

                # Avec suffixe — validation spéciale pour -na
                for sfx in sorted(SUFFIXES, key=len, reverse=True):
                    if reste.endswith(sfx) and len(reste) > len(sfx) + 1:
                        racine = reste[:-len(sfx)]

                        # Validation : si suffixe est -na, vérifie que
                        # la racine candidate existe vraiment dans le dico
                        if sfx == "na" and not suffixe_na_est_valide(forme, racine):
                            continue

                        if racine in DICTIONNAIRE:
                            candidats.append((racine, len(pfx) + len(sfx), "regles"))

                # Sans suffixe
                if reste in DICTIONNAIRE:
                    candidats.append((reste, len(pfx), "regles"))

                # Mutation après préfixe
                for racine_mutee in appliquer_mutations(reste):
                    candidats.append((racine_mutee, len(pfx) + 1, "mutation+prefixe"))

                # Suffixe + mutation
                for sfx in sorted(SUFFIXES, key=len, reverse=True):
                    if reste.endswith(sfx) and len(reste) > len(sfx) + 1:
                        reste_sans_sfx = reste[:-len(sfx)]
                        if sfx == "na" and not suffixe_na_est_valide(forme, reste_sans_sfx):
                            continue
                        for racine_mutee in appliquer_mutations(reste_sans_sfx):
                            candidats.append((racine_mutee, len(pfx) + len(sfx) + 1, "mutation+prefixe+suffixe"))

        # Sans préfixe, juste suffixe
        for sfx in sorted(SUFFIXES, key=len, reverse=True):
            if forme.endswith(sfx) and len(forme) > len(sfx) + 2:
                racine = forme[:-len(sfx)]

                # Validation spéciale pour -na
                if sfx == "na" and not suffixe_na_est_valide(forme, racine):
                    continue

                if racine in DICTIONNAIRE:
                    candidats.append((racine, len(sfx), "regles"))

        # ── N2 : mutation directe ──────────────────────────────────────
        for racine_mutee in appliquer_mutations(forme):
            candidats.append((racine_mutee, 1, "mutation_directe"))

    # ── Sélection du meilleur candidat ────────────────────────────────
    if candidats:
        meilleur = min(candidats, key=lambda x: len(x[0]))
        return {
            "racine":    meilleur[0],
            "methode":   meilleur[2],
            "confiance": "haute" if meilleur[0] in DICTIONNAIRE else "moyenne"
        }

    # ── N3 : forme restaurée seule (toetr → toetra) ───────────────────
    if restaure != mot_c and restaure in DICTIONNAIRE:
        return {"racine": restaure, "methode": "restauration", "confiance": "haute"}

    # ── N4 : mot dans le dico sans décomposition ──────────────────────
    if mot_c in DICTIONNAIRE:
        return {"racine": mot_c, "methode": "direct", "confiance": "haute"}

    # ── N5 : rien trouvé ──────────────────────────────────────────────
    return {"racine": mot_c, "methode": "non_trouvé", "confiance": "faible"}


# ── Test complet ───────────────────────────────────────────────────────
print("=== Test lemmatisation corrigée ===\n")
tests = [
    ("alina",    "alina"),   # -na ne doit PAS être retiré
    ("fomban",   "fomba"),   # -n liaison → restaure voyelle finale
    ("toetr",    "toetra"),  # apostrophe a coupé le -a final
    ("drazana",  "razana"),  # mutation d → r
    ("danja",    "lanja"),   # mutation d → l
    ("fitiavana","tia"),     # fi- + -ana
    ("mianatra", "anatra"),  # mi-
    ("malagasy", "malagasy"),# racine directe, pas de décomposition
]

print(f"{'Mot':<18} {'Lemme trouvé':<18} {'Attendu':<15} {'Méthode':<30} {'OK?'}")
print("-" * 85)
for mot, attendu in tests:
    r = lemmatiser(mot)
    ok = "✅" if r["racine"] == attendu else "⚠️ "
    print(f"{mot:<18} {r['racine']:<18} {attendu:<15} {r.get('methode','?'):<30} {ok}")

=== Test lemmatisation corrigée ===

Mot                Lemme trouvé       Attendu         Méthode                        OK?
-------------------------------------------------------------------------------------
alina              ali                alina           regles                         ⚠️ 
fomban             fomban             fomba           direct                         ⚠️ 
toetr              toetra             toetra          restauration                   ✅
drazana            razana             razana          mutation_directe               ✅
danja              lanja              lanja           mutation_directe               ✅
fitiavana          tiava              tia             regles                         ⚠️ 
mianatra           anatra             anatra          regles                         ✅
malagasy           malagasy           malagasy        direct                         ✅


**Exemple complet**

In [38]:
def analyser(texte: str) -> None:
    print("=" * 60)
    print(f"TEXTE : {texte}")
    print("=" * 60)

    tokens = tokeniser(texte)
    print(f"Tokens : {tokens}\n")

    vus = set()
    for mot in tokens:
        if mot in vus:
            continue
        vus.add(mot)

        if mot in STOPWORDS_MG:
            print(f"  ⏭️  '{mot}' — mot grammatical")
            continue

        print(f"\n  🔤 '{mot}'")

        # M1 : orthographe — utilise est_correct() corrigé
        if est_correct(mot):
            # Affiche la forme restaurée si différente
            restaure = restaurer_voyelle_finale(mot.lower())
            if restaure != mot.lower():
                print(f"     ✅ Correct (forme complète : '{restaure}')")
            else:
                print(f"     ✅ Correct")
        else:
            sugg = suggerer(mot)
            print(f"     ❌ Inconnu | Suggestions : {[s for s,_ in sugg] or ['—']}")

        # M2 : phonotactique
        for e in verifier_phonotactique(mot):
            print(f"     ⚠️  [{e.regle}] {e.description}")

        # M3 : lemmatisation
        r = lemmatiser(mot)
        if r["methode"] != "non_trouvé":
            print(f"     🌱 Lemme : '{r['racine']}' ({r['methode']})")


# ── Test ───────────────────────────────────────────────────────────────
analyser("Ny toetr'andro androany dia tsara")
analyser("Ny fomban-drazana malagasy dia manan-danja")
analyser("Mila mianatra ianareo")


TEXTE : Ny toetr'andro androany dia tsara
Tokens : ['ny', 'toetr', 'andro', 'androany', 'dia', 'tsara']

  ⏭️  'ny' — mot grammatical

  🔤 'toetr'
     ✅ Correct (forme complète : 'toetra')
     ⚠️  [FIN_CONSONNE] Fin de mot atypique (malagasy finit par voyelle ou -ny)
     🌱 Lemme : 'toetra' (restauration)

  🔤 'andro'
     ✅ Correct
     🌱 Lemme : 'andro' (direct)

  🔤 'androany'
     ✅ Correct
     🌱 Lemme : 'roa' (mutation+prefixe+suffixe)
  ⏭️  'dia' — mot grammatical

  🔤 'tsara'
     ✅ Correct
     🌱 Lemme : 'tsara' (direct)
TEXTE : Ny fomban-drazana malagasy dia manan-danja
Tokens : ['ny', 'fomban', 'drazana', 'malagasy', 'dia', 'manan', 'danja']

  ⏭️  'ny' — mot grammatical

  🔤 'fomban'
     ✅ Correct
     🌱 Lemme : 'fomban' (direct)

  🔤 'drazana'
     ✅ Correct
     🌱 Lemme : 'razana' (mutation_directe)

  🔤 'malagasy'
     ✅ Correct
     🌱 Lemme : 'malagasy' (direct)
  ⏭️  'dia' — mot grammatical

  🔤 'manan'
     ✅ Correct (forme complète : 'manao')
     🌱 Lemme : 'nan' 